# LU Partial Pivoting

Permutation convention: A[p, :] = L @ U. Solve uses b[p].

Run from the repository folder or any folder below it. Requires NumPy, SciPy and Numba.


In [1]:
import time
import sys
from pathlib import Path
import numpy as np
from scipy.sparse import csr_matrix, issparse

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside Numerical_Linear_Algebra_Algorithms")
sys.path.insert(0, str(ROOT))
from common import loader, data_path, lu_solve, relative_residual
from direct_methods.lu_partial_pivoting import lu_partial_pivoting


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


## 3×3


In [2]:
A = np.array([[0., 2., 1.], [1., 0., 3.], [4., 5., 6.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = lu_partial_pivoting(A)
result, seconds = timed(lu_partial_pivoting, A)
print("Seconds:", seconds)
L, U, p = result
x = lu_solve(L, U, b, p)
print("Factorization probe error:", float(np.linalg.norm(A[p] @ Z - L @ (U @ Z)) / np.linalg.norm(A[p] @ Z)))
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
assert relative_residual(A, x, b) < 1e-8


Seconds: 5.04200579598546e-06
Factorization probe error: 2.4033730387553828e-17
Relative residual: 0.0
Relative solution error: 0.0


## SuiteSparse

Change MATRIX to an existing file. bcsstk09 is 1,083×1,083; bcsstk17 is 10,974×10,974; west1505 is nonsymmetric and requires pivoting. No 10^5 matrix exists in this repo. Dense factorization costs O(n³); run the optional 10^4 case deliberately.


In [3]:
MATRIX = "west1505"
MAT_PATH = None
A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=True)
print("Matrix:", MATRIX, "shape:", A.shape)
x_true = np.ones(A.shape[0])
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))


Matrix: west1505 shape: (1505, 1505)


In [4]:
result, seconds = timed(lu_partial_pivoting, A)
print("Seconds:", seconds)
L, U, p = result
x = lu_solve(L, U, b, p)
print("Factorization probe error:", float(np.linalg.norm(A[p] @ Z - L @ (U @ Z)) / np.linalg.norm(A[p] @ Z)))
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))


Seconds: 1.0903487920004409
Factorization probe error: 1.1149316911674758e-16
Relative residual: 9.414870270109185e-17
Relative solution error: 1.4018832658800125e-09
